In [1]:
import os
import sys
import json
import math
import pandas as pd
import numpy as np
import random
import torch

import torch
from torch.utils.data import Dataset, DataLoader
from MinecraftVAEDataset import MinecraftVAEDataset, collate_fn
from text2mcVAE import text2mcVAE
import torch.nn as nn
from text2mc_train_utils import embedding_to_tokens

In [2]:
dataset = MinecraftVAEDataset(
    data_path='../../data/MinecraftVAEDataset/',
    sample_names=['batch_319_8281.npy', 'batch_225_5840.npy', "batch_2_48.npy"],
    dims=(24,24,24),
    mask_threshold=0
)

dataloader = DataLoader(
    dataset,
    batch_size=1,
    collate_fn=collate_fn
)

Added 0 blocks for potential tranformations
Added 0 blocks for potential tranformations


In [3]:
# checkpoint = torch.load('../../train_results/reduced_dataset/checkpoint_epoch_3.pth',map_location='cpu')
checkpoint_small_dataset = torch.load('../../train_results/small_dataset_roi_aug/checkpoint_epoch_6.pth',map_location='cpu')
checkpoint_text2mc = torch.load('../../train_results/checkpoint_epoch_43.pth',map_location='cpu')

model_small_dataset = text2mcVAE()
model_text2mc = text2mcVAE()

model_small_dataset.load_state_dict(checkpoint_small_dataset['model_state_dict'])
model_text2mc.load_state_dict(checkpoint_text2mc['model_state_dict'])

<All keys matched successfully>

In [7]:
embs = []
for index, i in enumerate(dataloader):
    embs.append(i[0].float())
    tok = i[1].numpy().squeeze()
    dataset.arr2schem(tok, './', f"{index}_orig")

Successfully saved schematic to 0_orig.schem
Successfully saved schematic to 1_orig.schem
Successfully saved schematic to 2_orig.schem


In [24]:
for i, emb in enumerate(embs):
    # z, mu, logvar, embeddings_pred, block_air_pred = model_text2mc(emb)
    # block_air_pred = block_air_pred.squeeze()
    # to_blocks = embedding_to_tokens(embeddings_pred.detach(), dataset.token2vector).squeeze().numpy()
    # to_blocks[block_air_pred < 0.6] = dataset.block2token["minecraft:air"]
    # dataset.arr2schem(to_blocks, './', f"{i}_text2mc")
    
    z, mu, logvar, embeddings_pred, block_air_pred = model_small_dataset(emb)
    block_air_pred = block_air_pred.squeeze()
    to_blocks = embedding_to_tokens(embeddings_pred.detach(), dataset.token2vector).squeeze().numpy()
    to_blocks[block_air_pred < 0.25] = dataset.block2token["minecraft:air"]
    dataset.arr2schem(to_blocks, './', f"{i}_small_dataset")

Successfully saved schematic to 0_small_dataset.schem
Successfully saved schematic to 1_small_dataset.schem
Successfully saved schematic to 2_small_dataset.schem


In [6]:
# print(f"Num parameters: {sum(p.numel() for p in model.parameters())}")
# z, mu, logvar, embeddings_pred, block_air_pred = model(sample)
# print(f"Latent Size: {z.shape}")